# s_spatioloji: Spatial Analysis

This notebook demonstrates spatial analysis capabilities using both **point-based** (centroid) and **polygon-based** (boundary) approaches.

**Prerequisites:** Run `01_basic_workflow.ipynb` first to have a processed dataset with clusters.

**What you'll learn:**
- Point-based analysis: KNN graphs, neighborhoods, spatial autocorrelation, Ripley's functions
- Polygon-based analysis: contact graphs, morphology, border enrichment
- Statistical testing: permutation tests, Clark-Evans, DCLF envelopes
- Visualization of all spatial results

In [ ]:
import matplotlib.pyplot as plt
from s_spatioloji.data.core import s_spatioloji as ssj

# Open processed dataset (from 01_basic_workflow)
ssp = ssj.open("/path/to/processed_dataset")
print(ssp)

---
# Part 1: Point-Based Spatial Analysis

Uses cell centroids (x, y coordinates). Scales to 50-100M cells via sparse matrix operations.

## 1.1 Build KNN Graph

The spatial KNN graph connects each cell to its k nearest neighbors based on Euclidean distance. This is the foundation for all point-based spatial analyses.

In [ ]:
from s_spatioloji.spatial.point.graph import build_knn_graph, build_radius_graph

# Build KNN graph (k=15 nearest neighbors)
build_knn_graph(ssp, k=15)

# Inspect the edge list
edges = ssp.maps["knn_graph"].compute()
print(f"Number of edges: {len(edges):,}")
print(f"Mean distance: {edges['distance'].mean():.1f} um")
edges.head()

In [ ]:
# Alternatively, build a radius-based graph (fixed distance threshold)
# build_radius_graph(ssp, radius=50.0, output_key="radius_graph")

## 1.2 Neighborhood Analysis

Quantify the cellular microenvironment around each cell.

In [ ]:
from s_spatioloji.spatial.point.neighborhoods import (
    neighborhood_composition,
    neighborhood_diversity,
    nth_order_neighbors,
)

# Neighborhood composition: count neighbor types per cell
neighborhood_composition(ssp, cluster_key="leiden", graph_key="knn_graph")

comp = ssp.maps["pt_nhood_composition"].compute()
print(f"Composition table: {comp.shape[0]} cells x {comp.shape[1] - 1} clusters")
comp.head()

In [ ]:
# Weighted by inverse distance (closer neighbors contribute more)
neighborhood_composition(
    ssp, cluster_key="leiden", weighted=True,
    output_key="pt_nhood_composition_weighted",
)

In [ ]:
# Neighborhood diversity: Shannon entropy + Gini-Simpson index
neighborhood_diversity(ssp, cluster_key="leiden")

div = ssp.maps["pt_nhood_diversity"].compute()
print(f"Mean Shannon entropy: {div['shannon'].mean():.3f}")
print(f"Mean Simpson index:   {div['simpson'].mean():.3f}")
div.head()

In [ ]:
# Nth-order neighbors: count cells reachable at each hop distance
nth_order_neighbors(ssp, order=3)

nth = ssp.maps["pt_nhood_nth_order"].compute()
print(f"Mean 1st-order neighbors: {nth['n_order_1'].mean():.1f}")
print(f"Mean 2nd-order neighbors: {nth['n_order_2'].mean():.1f}")
print(f"Mean 3rd-order neighbors: {nth['n_order_3'].mean():.1f}")

## 1.3 Spatial Autocorrelation

Test whether features are spatially clustered, dispersed, or random.

In [ ]:
from s_spatioloji.spatial.point.patterns import morans_i, gearys_c, getis_ord_gi

# Moran's I on cluster labels
# I > 0 = spatially clustered, I < 0 = dispersed, I ~ 0 = random
morans_i(ssp, feature_key="leiden")

mi = ssp.maps["pt_morans_i"].compute()
print("Moran's I results:")
mi

In [ ]:
# Geary's C: complementary to Moran's I
# C < 1 = clustered, C = 1 = random, C > 1 = dispersed
gearys_c(ssp, feature_key="leiden")

gc = ssp.maps["pt_gearys_c"].compute()
print("Geary's C results:")
gc

In [ ]:
# Getis-Ord Gi*: identify local hotspots for a numeric feature
# High z-scores = hot spots, low z-scores = cold spots
getis_ord_gi(ssp, feature_key="leiden")

gi = ssp.maps["pt_getis_ord"].compute()
print(f"Gi* z-score range: [{gi['gi_stat'].min():.2f}, {gi['gi_stat'].max():.2f}]")
print(f"Significant cells (p < 0.05): {(gi['p_value'] < 0.05).sum():,}")

In [ ]:
# Visualize Gi* hotspots on spatial map
from s_spatioloji.visualization.spatial import spatial_scatter

spatial_scatter(ssp, color_by="pt_getis_ord", cmap="RdBu_r", point_size=0.5, figsize=(12, 12))
plt.title("Getis-Ord Gi* Hotspot Map")
plt.show()

## 1.4 Colocalization

Test whether cell type pairs are found together more (or less) than expected by chance.

In [ ]:
from s_spatioloji.spatial.point.patterns import colocalization
from s_spatioloji.visualization.analysis import colocalization_heatmap

# Compute colocalization
colocalization(ssp, cluster_key="leiden")

coloc = ssp.maps["pt_colocalization"].compute()
print(f"{len(coloc)} cluster pairs analyzed")
coloc.head(10)

In [ ]:
# Visualize as heatmap (log2 ratio: positive = co-localized, negative = segregated)
colocalization_heatmap(ssp, metric="log2_ratio", cmap="RdBu_r")
plt.title("Colocalization (log2 observed/expected)")
plt.show()

## 1.5 Neighborhood Composition Visualization

In [ ]:
from s_spatioloji.visualization.analysis import neighborhood_bar

# Stacked bar chart: mean neighbor composition per cluster
neighborhood_bar(ssp, maps_key="pt_nhood_composition", cluster_key="leiden", normalize=True)
plt.title("Neighborhood Composition by Cluster")
plt.show()

## 1.6 Ripley's Functions

Characterize spatial point patterns: clustering, regularity, or complete spatial randomness (CSR).

In [ ]:
from s_spatioloji.spatial.point.ripley import ripley_k, ripley_l, ripley_g
from s_spatioloji.visualization.analysis import ripley_plot

# Ripley's K: K(r) > pi*r^2 means clustering at scale r
ripley_k(ssp, n_radii=30, cluster_key="leiden")

ripley_plot(ssp, function="K")
plt.title("Ripley's K Function")
plt.show()

In [ ]:
# Ripley's L (variance-stabilized K): L(r) > 0 = clustered, L(r) < 0 = dispersed
ripley_l(ssp, n_radii=30, cluster_key="leiden")

ripley_plot(ssp, function="L")
plt.title("Ripley's L Function (L > 0 = clustered)")
plt.show()

In [ ]:
# Ripley's G (nearest-neighbor): G(r) > G_theo = clustered
ripley_g(ssp, n_radii=30)

ripley_plot(ssp, function="G")
plt.title("Ripley's G Function (Nearest Neighbor)")
plt.show()

## 1.7 Statistical Testing

In [ ]:
from s_spatioloji.spatial.point.statistics import (
    clark_evans,
    dclf_envelope,
    permutation_test,
    quadrat_density,
)

# Clark-Evans index: R < 1 = clustered, R = 1 = random, R > 1 = dispersed
clark_evans(ssp, cluster_key="leiden")

ce = ssp.maps["pt_clark_evans"].compute()
print("Clark-Evans nearest-neighbor index:")
ce

In [ ]:
# Permutation test for colocalization significance
permutation_test(ssp, cluster_key="leiden", n_permutations=199)

perm = ssp.maps["pt_permutation_test"].compute()
sig = perm[perm["p_value"] < 0.05]
print(f"Significant pairs (p < 0.05): {len(sig)} / {len(perm)}")
sig.sort_values("z_score", ascending=False).head(10)

In [ ]:
# Quadrat density: chi-squared test for spatial uniformity
quadrat_density(ssp, cluster_key="leiden", n_bins=10)

qd = ssp.maps["pt_quadrat_density"].compute()
print("Quadrat density results:")
qd

In [ ]:
from s_spatioloji.visualization.analysis import envelope_plot

# DCLF envelope test: is the observed pattern significantly different from CSR?
dclf_envelope(ssp, function="K", n_simulations=99, n_radii=30)

envelope_plot(ssp)
plt.title("DCLF Envelope Test (K function)")
plt.show()

---
# Part 2: Polygon-Based Spatial Analysis

Uses cell boundary geometries for topology-aware analysis. Requires `boundaries.parquet`.

In [ ]:
if not ssp.has_boundaries:
    print("No boundaries available. Skipping polygon-based analysis.")
    print("Polygon analysis requires cell/nucleus boundaries from segmentation.")
else:
    print(f"Boundaries available for {ssp.boundaries.n_cells:,} cells")

## 2.1 Contact Graph

Build a graph where edges connect physically touching cells (shared boundary).

In [ ]:
from s_spatioloji.spatial.polygon.graph import build_contact_graph

# Build contact graph (buffer_distance=0 for strict touching)
build_contact_graph(ssp, buffer_distance=0.0)

contacts = ssp.maps["contact_graph"].compute()
print(f"Contact edges: {len(contacts):,}")
print(f"Mean shared boundary length: {contacts['shared_length'].mean():.1f} um")
contacts.head()

## 2.2 Cell Morphology

Compute 13 shape descriptors for each cell boundary.

In [ ]:
from s_spatioloji.spatial.polygon.morphology import cell_morphology

cell_morphology(ssp)

morph = ssp.maps["morphology"].compute()
print(f"Morphology metrics: {[c for c in morph.columns if c != 'cell_id']}")
morph.describe()

## 2.3 Polygon Neighborhoods + Patterns

In [ ]:
from s_spatioloji.spatial.polygon.neighborhoods import (
    neighborhood_composition as poly_nhood_comp,
    neighborhood_diversity as poly_nhood_div,
)
from s_spatioloji.spatial.polygon.patterns import (
    border_enrichment,
    colocalization as poly_coloc,
    morans_i as poly_morans,
)

# Neighborhood composition (polygon-based: direct physical neighbors)
poly_nhood_comp(ssp, cluster_key="leiden")

# Border enrichment: which clusters are over-represented at tissue borders?
border_enrichment(ssp, cluster_key="leiden")

be = ssp.maps["border_enrichment"].compute()
print("Border enrichment by cluster:")
be

## 2.4 Polygon Visualization

In [ ]:
from s_spatioloji.visualization.spatial import spatial_polygons

# Full tissue view with polygon boundaries
spatial_polygons(
    ssp, color_by="leiden",
    edgecolor="black", linewidth=0.1, alpha=0.7,
    figsize=(14, 14),
)
plt.title("Cell Boundaries Colored by Cluster")
plt.show()

In [ ]:
# Zoomed region with polygon boundaries
spatial_polygons(
    ssp, color_by="leiden",
    xlim=(500, 1500), ylim=(500, 1500),
    edgecolor="black", linewidth=0.3, alpha=0.8,
    figsize=(10, 10),
)
plt.title("Zoomed Region")
plt.show()

---
# Part 3: Comparing Point vs Polygon Analysis

The two approaches complement each other:
- **Point-based**: fast, scales to 100M cells, distance-weighted neighborhoods
- **Polygon-based**: topology-aware (contact = touching), captures boundary morphology

In [ ]:
# Compare point vs polygon Moran's I
if ssp.has_boundaries:
    poly_morans(ssp, feature_key="leiden")
    
    mi_point = ssp.maps["pt_morans_i"].compute()
    mi_poly = ssp.maps["morans_i"].compute()
    
    print("Moran's I comparison:")
    print(f"  Point-based (KNN):     I = {mi_point['I'].iloc[0]:.4f}, p = {mi_point['p_value'].iloc[0]:.2e}")
    print(f"  Polygon-based (contact): I = {mi_poly['I'].iloc[0]:.4f}, p = {mi_poly['p_value'].iloc[0]:.2e}")

---
# Part 4: Working with Images at Different Pyramid Levels

When overlaying cells on downsampled images, use `ImageCollection.scale_coordinates` to match coordinate systems.

In [ ]:
if ssp.has_images:
    # Show available pyramid levels
    store = ssp.images[ssp.images.default_image]
    print(f"Image: {ssp.images.default_image}")
    print(f"Pyramid levels: {store.n_levels}")
    for lvl in range(min(store.n_levels, 8)):
        shape = store.level_shape(lvl)
        ps = ssp.images.pixel_size_at(lvl)
        print(f"  Level {lvl}: {shape}, pixel size = {ps:.4f} um/px")

In [ ]:
if ssp.has_images:
    # Scale cell coordinates to match a specific pyramid level
    cells_df = ssp.cells.df[["cell_id", "x", "y"]].compute()
    
    # Level 0 (full resolution)
    scaled_l0 = ssp.images.scale_coordinates(cells_df, level=0)
    print(f"Level 0 - x range: [{scaled_l0['x'].min():.0f}, {scaled_l0['x'].max():.0f}] px")
    
    # Level 3 (8x downsampled)
    scaled_l3 = ssp.images.scale_coordinates(cells_df, level=3)
    print(f"Level 3 - x range: [{scaled_l3['x'].min():.0f}, {scaled_l3['x'].max():.0f}] px")

In [ ]:
if ssp.has_images:
    # Example: overlay cells on a downsampled image thumbnail
    level = 4  # 16x downsampled, manageable for display
    
    store = ssp.images[ssp.images.default_image]
    thumbnail = store.to_dask(level=level).compute()
    
    # Scale coordinates to match this level
    scaled = ssp.images.scale_coordinates(cells_df, level=level)
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Show image (take first channel if multi-channel)
    if thumbnail.ndim == 3:
        ax.imshow(thumbnail[0], cmap="gray", alpha=0.6)
    else:
        ax.imshow(thumbnail, cmap="gray", alpha=0.6)
    
    # Overlay cells
    ax.scatter(scaled["x"], scaled["y"], s=0.1, c="red", alpha=0.3)
    ax.set_title(f"Cells on image (level {level}, {ssp.images.pixel_size_at(level):.1f} um/px)")
    plt.show()

## Summary

This notebook demonstrated:

**Point-based analysis:**
- KNN/radius graph construction
- Neighborhood composition, diversity, nth-order neighbors
- Spatial autocorrelation (Moran's I, Geary's C, Getis-Ord Gi*)
- Colocalization analysis
- Ripley's K/L/G functions
- Statistical tests (Clark-Evans, permutation, quadrat density, DCLF)

**Polygon-based analysis:**
- Contact graph from cell boundaries
- Cell morphology descriptors
- Border enrichment
- Polygon-based neighborhoods and spatial autocorrelation

**Image integration:**
- Pyramid level access
- Coordinate scaling for overlay at any resolution

All results are persisted to `maps/` with `pt_` prefix (point) or no prefix (polygon).